# STARRT 6A (fixed): Phi-3.5 zero-shot, model split over 2 GPUs (--device_map auto) to avoid CUDA out of memory.

Add Input: **starrt-gpu6**. GPU T4 x2, Internet On. Save & Run All (Commit).

# STARRT round 6A: second LLM family (Phi-3.5-mini, zero-shot)

**Settings:** GPU T4 x2, Internet On, Add Input → `starrt-gpu6`. Run the first two cells, then **Save Version → Save & Run All (Commit)**. About 2-3 hours. Results → `starrt_gpu6A_results.zip`.

In [ ]:
import glob, os, subprocess
hits = glob.glob('/kaggle/input/**/run_llm_zeroshot.py', recursive=True)
assert hits, 'Dataset not attached: Add Input -> starrt-gpu6'
CODE = os.path.dirname(hits[0]); print('code folder:', CODE)
import torch; print('GPUs:', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

def run(script, args=''):
    cmd = f'cd /kaggle/working && python -u {CODE}/{script} {args}'
    print('>>', cmd, flush=True); r = subprocess.run(cmd, shell=True)
    if r.returncode != 0: print('!! FAILED:', script, args, '(continuing with the next step)')

## 0. Smoke test (3 min): must print a [smoke_phi ...] line with acc well above 0

In [ ]:
run('run_llm_zeroshot.py', '--model microsoft/Phi-3.5-mini-instruct --end_text "<|end|>" --datasets faruque --folds 1 --limit 12 --tag smoke_phi')

## A. Phi-3.5-mini zero-shot on the same samples of all five folds (2,400 Djinni + 1,057 Faruque rows)

In [ ]:
run('run_llm_zeroshot.py', '--model microsoft/Phi-3.5-mini-instruct --end_text "<|end|>" --device_map auto --datasets djinni,faruque --folds 0,1,2,3,4 --per_class 40 --tag llm_phi3.5-mini_zeroshot')
# if this step prints 'non-finite scores', run instead (slower):
# run('run_llm_zeroshot.py', '--model microsoft/Phi-3.5-mini-instruct --end_text "<|end|>" --dtype float32 --device_map auto --datasets djinni,faruque --folds 0,1,2,3,4 --per_class 40 --tag llm_phi3.5-mini_zeroshot')

## Zip results

In [ ]:
import shutil
for f in glob.glob('/kaggle/working/results/smoke*'): os.remove(f)
shutil.make_archive('/kaggle/working/starrt_gpu6A_results', 'zip', '/kaggle/working/results')
print(sorted(os.listdir('/kaggle/working/results')))